
# Triadic Cell Notebook v24
## DDD-style recurrent lattice runtime

This notebook codes the next step directly.

The working compression is:

- **inner CPU** = local packet / boundary closure cell
- **outer CPU** = bridge policy over those cells
- **lattice** = all local cells updating in parallel
- **value** = late readout only

This is the DDD-like version of the same law:

- **Domain / Aggregate**
  - local packet cell
  - holds persistent state and invariants

- **Application / Orchestration**
  - bridge policy over the packet field

- **Read model**
  - value projection emitted after closure

## What this notebook does

1. Builds a recurrent 2D lattice of packet cells.
2. Gives each cell persistent local state:
   - tail
   - boundary_parallel
   - boundary_perp
   - carry
   - closure
   - trace
3. Updates all cells in parallel across time.
4. Trains a small bridge head from synthetic traces.
5. Compares:
   - heuristic bridge CPU
   - learned bridge CPU
6. Reports lattice-level metrics:
   - class accuracy
   - closure
   - carry error
   - promote / retire / repair rates

This is still **not** base-model training.
This is the runtime.



## Practical meaning

This notebook affects the software stack around the model:

- **LLM / foundation model**
  - still just a proposal engine

- **Shape runtime**
  - reads the packet state field

- **Bridge policy**
  - the trainable controller
  - decides what happens next for each cell

- **Value layer**
  - late readout only

So this is a recurrent, nested-CPU version of the bridge runtime.


In [ ]:

from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(suppress=True, precision=4)

rng = np.random.default_rng(42)

print("Environment ready.")



## Canonical packet / tail world


In [ ]:

prefixes = np.array([
    [1.00, 0.05, 0.05, 0.05],
    [0.05, 1.00, 0.05, 0.05],
    [0.05, 0.05, 1.00, 0.05],
    [0.05, 0.05, 0.05, 1.00],
], dtype=float)

tail_prototypes = np.array([
    [0.92, 0.18, 0.58, 0.22],
    [0.18, 0.90, 0.26, 0.60],
    [0.62, 0.24, 0.90, 0.18],
    [0.22, 0.62, 0.18, 0.88],
], dtype=float)

ACTION_NAMES = ["promote", "hold", "retire", "grow_left", "grow_right", "grow_carry"]
ACTION_TO_ID = {name: i for i, name in enumerate(ACTION_NAMES)}
ID_TO_ACTION = {i: name for name, i in ACTION_TO_ID.items()}

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    na = np.linalg.norm(a) + 1e-8
    nb = np.linalg.norm(b) + 1e-8
    return float(np.dot(a, b) / (na * nb))

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def one_hot(y, k):
    out = np.zeros((len(y), k), dtype=float)
    out[np.arange(len(y)), y] = 1.0
    return out

def top2_margin(scores: np.ndarray):
    order = np.argsort(scores)[::-1]
    top1_idx = int(order[0])
    top1 = float(scores[order[0]])
    top2 = float(scores[order[1]]) if len(order) > 1 else 0.0
    return top1_idx, top1, top1 - top2



## Corruption generator


In [ ]:

MODES = [
    "clean",
    "missing_left",
    "missing_right",
    "carry_fracture",
    "bad_carry",
    "ambiguous",
    "noisy",
    "broken",
    "borderline",
    "wrong_class",
]

MODE_WEIGHTS = np.array([0.10, 0.12, 0.12, 0.12, 0.12, 0.12, 0.08, 0.06, 0.08, 0.08], dtype=float)
MODE_WEIGHTS = MODE_WEIGHTS / MODE_WEIGHTS.sum()

def corrupt_tail(base_tail: np.ndarray, mode: str, target_slot: int) -> np.ndarray:
    x = base_tail.copy()

    if mode == "clean":
        x += rng.normal(0.0, 0.02, size=x.shape[0])

    elif mode == "missing_left":
        x[0] *= rng.uniform(0.0, 0.18)
        x += rng.normal(0.0, 0.02, size=x.shape[0])

    elif mode == "missing_right":
        x[3] *= rng.uniform(0.0, 0.18)
        x += rng.normal(0.0, 0.02, size=x.shape[0])

    elif mode == "carry_fracture":
        x[1] = np.clip(x[1] + rng.uniform(0.35, 0.60), 0.0, 1.0)
        x[2] = np.clip(x[2] + rng.normal(0.0, 0.01), 0.0, 1.0)
        x[0] = np.clip(x[0] + rng.normal(0.0, 0.01), 0.0, 1.0)
        x[3] = np.clip(x[3] + rng.normal(0.0, 0.01), 0.0, 1.0)

    elif mode == "bad_carry":
        x[1] = np.clip(x[1] + rng.uniform(0.22, 0.48), 0.0, 1.0)
        x[2] = np.clip(x[2] - rng.uniform(0.08, 0.22), 0.0, 1.0)
        x += rng.normal(0.0, 0.015, size=x.shape[0])

    elif mode == "ambiguous":
        alt = int((target_slot + rng.integers(1, 4)) % len(tail_prototypes))
        x = 0.52 * base_tail + 0.48 * tail_prototypes[alt] + rng.normal(0.0, 0.02, size=x.shape[0])

    elif mode == "noisy":
        mask = rng.random(x.shape[0]) < 0.25
        x[mask] = 0.0
        x += rng.normal(0.0, 0.05, size=x.shape[0])

    elif mode == "broken":
        x = rng.uniform(0.0, 1.0, size=x.shape[0])

    elif mode == "borderline":
        x = 0.85 * base_tail + 0.15 * np.roll(base_tail, 1) + rng.normal(0.0, 0.025, size=x.shape[0])
        x[0] = np.clip(x[0] - 0.08, 0.0, 1.0)
        x[3] = np.clip(x[3] - 0.08, 0.0, 1.0)

    elif mode == "wrong_class":
        alt = int((target_slot + rng.integers(1, 4)) % len(tail_prototypes))
        x = 0.10 * base_tail + 0.90 * tail_prototypes[alt] + rng.normal(0.0, 0.02, size=x.shape[0])

    return np.clip(x, 0.0, 1.0)



## Lattice state

Each cell holds persistent recurrent state.

- `tail`
- `boundary_parallel`
- `boundary_perp`
- `carry`
- `closure`
- `trace`


In [ ]:

def init_lattice(height: int = 8, width: int = 8):
    truth = rng.integers(0, 4, size=(height, width))
    modes = rng.choice(MODES, size=(height, width), p=MODE_WEIGHTS)

    tail = np.zeros((height, width, 4), dtype=float)
    boundary_parallel = np.zeros((height, width), dtype=float)
    boundary_perp = np.zeros((height, width), dtype=float)
    carry = np.zeros((height, width), dtype=float)
    closure = np.zeros((height, width), dtype=float)
    trace = np.zeros((height, width), dtype=float)

    for i in range(height):
        for j in range(width):
            cls = int(truth[i, j])
            t = corrupt_tail(tail_prototypes[cls], modes[i, j], cls)
            tail[i, j] = t
            boundary_parallel[i, j] = 0.5 * (t[0] + t[3])
            boundary_perp[i, j] = 0.5 * (t[1] + t[2])
            carry[i, j] = 0.5 * t[1] + 0.5 * t[2]
            closure[i, j] = cosine_similarity(t, tail_prototypes[cls])
            trace[i, j] = 0.0

    return {
        "truth": truth,
        "modes": modes,
        "tail": tail,
        "boundary_parallel": boundary_parallel,
        "boundary_perp": boundary_perp,
        "carry": carry,
        "closure": closure,
        "trace": trace,
    }



## Shape extraction for a cell

This is the inner CPU readout that the outer CPU will govern.


In [ ]:

def neighbor_tail_mean(state: dict, i: int, j: int) -> np.ndarray:
    H, W, _ = state["tail"].shape
    neighbors = []
    for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
        ni, nj = i + di, j + dj
        if 0 <= ni < H and 0 <= nj < W:
            neighbors.append(state["tail"][ni, nj])
    if len(neighbors) == 0:
        return np.zeros(4, dtype=float)
    return np.mean(np.stack(neighbors, axis=0), axis=0)

def predict_slot_from_tail(tail_vec: np.ndarray):
    scores = np.array([cosine_similarity(tail_vec, t) for t in tail_prototypes], dtype=float)
    pred_slot, _, margin = top2_margin(scores)
    return pred_slot, float(margin), scores

def packet_shape_features(state: dict, i: int, j: int):
    tail_vec = state["tail"][i, j]
    pred_slot, route_margin, _ = predict_slot_from_tail(tail_vec)

    candidate_tail = tail_prototypes[pred_slot]
    candidate_prefix = prefixes[pred_slot]
    neigh = neighbor_tail_mean(state, i, j)

    left_support = 1.0 - abs(tail_vec[0] - candidate_tail[0])
    hinge_fit = 1.0 - abs(tail_vec[2] - candidate_tail[2])
    right_support = 1.0 - abs(tail_vec[3] - candidate_tail[3])

    carry_target = 0.5 * candidate_tail[1] + 0.5 * candidate_tail[2]
    carry_fit = 1.0 - abs(state["carry"][i, j] - carry_target)

    tail_fit = cosine_similarity(tail_vec, candidate_tail)
    prefix_hint = cosine_similarity(tail_vec, candidate_prefix)
    contradiction = 1.0 - prefix_hint
    bilateral = 0.5 * (left_support + right_support)
    support_balance = 1.0 - abs(left_support - right_support)
    neighbor_agreement = cosine_similarity(tail_vec, neigh) if np.linalg.norm(neigh) > 0 else 0.0

    return {
        "pred_slot": int(pred_slot),
        "route_margin": float(route_margin),
        "left_support": float(np.clip(left_support, 0.0, 1.0)),
        "hinge_fit": float(np.clip(hinge_fit, 0.0, 1.0)),
        "right_support": float(np.clip(right_support, 0.0, 1.0)),
        "carry_fit": float(np.clip(carry_fit, 0.0, 1.0)),
        "tail_fit": float(np.clip(tail_fit, 0.0, 1.0)),
        "prefix_hint": float(np.clip(prefix_hint, 0.0, 1.0)),
        "contradiction": float(np.clip(contradiction, 0.0, 1.0)),
        "bilateral": float(np.clip(bilateral, 0.0, 1.0)),
        "support_balance": float(np.clip(support_balance, 0.0, 1.0)),
        "neighbor_agreement": float(np.clip(neighbor_agreement, 0.0, 1.0)),
        "boundary_parallel": float(np.clip(state["boundary_parallel"][i, j], 0.0, 1.0)),
        "boundary_perp": float(np.clip(state["boundary_perp"][i, j], 0.0, 1.0)),
        "closure": float(np.clip(state["closure"][i, j], 0.0, 1.0)),
        "trace": float(np.clip(state["trace"][i, j], 0.0, 1.0)),
    }

shape_cols = [
    "left_support",
    "hinge_fit",
    "right_support",
    "carry_fit",
    "tail_fit",
    "prefix_hint",
    "contradiction",
    "bilateral",
    "support_balance",
    "neighbor_agreement",
    "boundary_parallel",
    "boundary_perp",
    "closure",
    "trace",
    "route_margin",
]



## Bridge oracle

This is the heuristic outer CPU.


In [ ]:

def packet_gate(shape: dict):
    gate_score = (
        0.15 * shape["left_support"]
        + 0.18 * shape["hinge_fit"]
        + 0.15 * shape["right_support"]
        + 0.14 * shape["carry_fit"]
        + 0.10 * shape["tail_fit"]
        + 0.08 * shape["bilateral"]
        + 0.06 * shape["support_balance"]
        + 0.06 * shape["neighbor_agreement"]
        + 0.05 * shape["boundary_parallel"]
        + 0.03 * shape["boundary_perp"]
    )
    passes = (
        shape["left_support"] > 0.58 and
        shape["hinge_fit"] > 0.58 and
        shape["right_support"] > 0.58 and
        shape["carry_fit"] > 0.58 and
        shape["route_margin"] > 0.03 and
        gate_score > 0.62
    )
    return float(gate_score), bool(passes)

def bridge_oracle(shape: dict, truth_slot: int):
    gate_score, passes = packet_gate(shape)

    if (
        shape["carry_fit"] < 0.72 and
        shape["bilateral"] > 0.72 and
        shape["pred_slot"] == truth_slot
    ):
        return "grow_carry", gate_score, passes

    if shape["left_support"] < 0.54 and shape["right_support"] >= 0.60:
        return "grow_left", gate_score, passes

    if shape["right_support"] < 0.54 and shape["left_support"] >= 0.60:
        return "grow_right", gate_score, passes

    if shape["pred_slot"] != truth_slot and shape["tail_fit"] < 0.84 and shape["route_margin"] > 0.08:
        return "retire", gate_score, passes

    if shape["hinge_fit"] < 0.58 or shape["route_margin"] < 0.05 or gate_score < 0.60:
        return "hold", gate_score, passes

    if passes and shape["pred_slot"] == truth_slot:
        return "promote", gate_score, passes

    return "hold", gate_score, passes



## Inner CPU update law

The inner CPU updates all local cell states from the chosen bridge action.


In [ ]:

def apply_action(state: dict, i: int, j: int, action: str):
    tail_vec = state["tail"][i, j].copy()
    pred_slot, _, _ = predict_slot_from_tail(tail_vec)
    target = tail_prototypes[pred_slot].copy()
    neigh = neighbor_tail_mean(state, i, j)

    if action == "promote":
        tail_vec = 0.78 * tail_vec + 0.18 * target + 0.04 * neigh
        state["closure"][i, j] = np.clip(0.88 * state["closure"][i, j] + 0.12 * cosine_similarity(tail_vec, target), 0.0, 1.0)
        state["trace"][i, j] = np.clip(0.90 * state["trace"][i, j] + 0.10 * 1.0, 0.0, 1.0)

    elif action == "hold":
        tail_vec = 0.86 * tail_vec + 0.10 * neigh + 0.04 * target
        state["closure"][i, j] = np.clip(0.94 * state["closure"][i, j] + 0.06 * cosine_similarity(tail_vec, target), 0.0, 1.0)
        state["trace"][i, j] = np.clip(0.92 * state["trace"][i, j] + 0.08 * 0.5, 0.0, 1.0)

    elif action == "retire":
        tail_vec = 0.55 * tail_vec + 0.20 * rng.uniform(0.0, 1.0, size=4) + 0.25 * neigh
        state["closure"][i, j] = np.clip(0.80 * state["closure"][i, j], 0.0, 1.0)
        state["trace"][i, j] = np.clip(0.90 * state["trace"][i, j] + 0.10 * 0.2, 0.0, 1.0)

    elif action == "grow_left":
        tail_vec[0] = np.clip(0.80 * tail_vec[0] + 0.20 * target[0], 0.0, 1.0)
        tail_vec[1:] = np.clip(0.92 * tail_vec[1:] + 0.08 * target[1:], 0.0, 1.0)
        state["closure"][i, j] = np.clip(0.92 * state["closure"][i, j] + 0.08 * cosine_similarity(tail_vec, target), 0.0, 1.0)
        state["trace"][i, j] = np.clip(0.90 * state["trace"][i, j] + 0.10 * 0.7, 0.0, 1.0)

    elif action == "grow_right":
        tail_vec[3] = np.clip(0.80 * tail_vec[3] + 0.20 * target[3], 0.0, 1.0)
        tail_vec[:3] = np.clip(0.92 * tail_vec[:3] + 0.08 * target[:3], 0.0, 1.0)
        state["closure"][i, j] = np.clip(0.92 * state["closure"][i, j] + 0.08 * cosine_similarity(tail_vec, target), 0.0, 1.0)
        state["trace"][i, j] = np.clip(0.90 * state["trace"][i, j] + 0.10 * 0.7, 0.0, 1.0)

    elif action == "grow_carry":
        carry_target = 0.5 * target[1] + 0.5 * target[2]
        state["carry"][i, j] = np.clip(0.65 * state["carry"][i, j] + 0.35 * carry_target, 0.0, 1.0)
        tail_vec[1] = np.clip(0.75 * tail_vec[1] + 0.25 * target[1], 0.0, 1.0)
        tail_vec[2] = np.clip(0.75 * tail_vec[2] + 0.25 * target[2], 0.0, 1.0)
        state["closure"][i, j] = np.clip(0.93 * state["closure"][i, j] + 0.07 * cosine_similarity(tail_vec, target), 0.0, 1.0)
        state["trace"][i, j] = np.clip(0.90 * state["trace"][i, j] + 0.10 * 0.8, 0.0, 1.0)

    state["tail"][i, j] = np.clip(tail_vec, 0.0, 1.0)
    state["boundary_parallel"][i, j] = 0.5 * (state["tail"][i, j, 0] + state["tail"][i, j, 3])
    state["boundary_perp"][i, j] = 0.5 * (state["tail"][i, j, 1] + state["tail"][i, j, 2])
    state["carry"][i, j] = np.clip(0.60 * state["carry"][i, j] + 0.40 * (0.5 * state["tail"][i, j, 1] + 0.5 * state["tail"][i, j, 2]), 0.0, 1.0)

def lattice_step_with_actions(state: dict, action_grid: np.ndarray):
    H, W = action_grid.shape
    for i in range(H):
        for j in range(W):
            apply_action(state, i, j, str(action_grid[i, j]))
    return state



## Trace generation for bridge-head training

We collect local shape/action examples from many small random lattices.


In [ ]:

def collect_trace_dataset(num_worlds: int = 120, steps_per_world: int = 8, height: int = 6, width: int = 6):
    rows = []
    for _ in range(num_worlds):
        state = init_lattice(height=height, width=width)

        for step in range(steps_per_world):
            H, W = state["truth"].shape
            action_grid = np.empty((H, W), dtype=object)

            for i in range(H):
                for j in range(W):
                    shape = packet_shape_features(state, i, j)
                    action, gate_score, packet_pass = bridge_oracle(shape, int(state["truth"][i, j]))

                    row = {col: shape[col] for col in shape_cols}
                    row["truth_slot"] = int(state["truth"][i, j])
                    row["pred_slot"] = int(shape["pred_slot"])
                    row["action"] = action
                    row["action_id"] = ACTION_TO_ID[action]
                    row["gate_score"] = gate_score
                    row["packet_pass"] = int(packet_pass)
                    rows.append(row)

                    action_grid[i, j] = action

            state = lattice_step_with_actions(state, action_grid)

    return pd.DataFrame(rows)

trace_df = collect_trace_dataset(num_worlds=120, steps_per_world=8, height=6, width=6)
trace_df.head()



## Trace distribution


In [ ]:

trace_summary = pd.DataFrame([{
    "trace_rows": int(len(trace_df)),
    "mean_gate_score": float(trace_df["gate_score"].mean()),
    "packet_pass_rate": float(trace_df["packet_pass"].mean()),
}])

trace_action_mix = (
    trace_df["action"]
    .value_counts(normalize=True)
    .rename_axis("action")
    .reset_index(name="fraction")
)

trace_summary, trace_action_mix



## Train a tiny bridge head

This is the outer CPU learner.

It predicts bridge actions from local recurrent shape state.


In [ ]:

def standardize_train_test(X_train, X_test):
    mu = X_train.mean(axis=0, keepdims=True)
    sd = X_train.std(axis=0, keepdims=True) + 1e-8
    return (X_train - mu) / sd, (X_test - mu) / sd, mu, sd

def train_multiclass_logreg(X, y, sample_weight=None, lr=0.08, epochs=800):
    n_samples, n_features = X.shape
    n_classes = int(np.max(y)) + 1

    W = np.zeros((n_features, n_classes), dtype=float)
    b = np.zeros((1, n_classes), dtype=float)

    Y = one_hot(y, n_classes)

    if sample_weight is None:
        sample_weight = np.ones(n_samples, dtype=float)

    sample_weight = sample_weight.reshape(-1, 1)
    weight_scale = sample_weight / (sample_weight.sum() + 1e-8)

    loss_history = []

    for _ in range(epochs):
        logits = X @ W + b
        probs = softmax(logits)

        loss = -np.sum(weight_scale * np.sum(Y * np.log(probs + 1e-8), axis=1, keepdims=True))
        loss_history.append(float(loss))

        dlogits = (probs - Y) * weight_scale
        dW = X.T @ dlogits
        db = dlogits.sum(axis=0, keepdims=True)

        W -= lr * dW
        b -= lr * db

    return W, b, loss_history

def predict_logreg(X, W, b):
    probs = softmax(X @ W + b)
    return probs.argmax(axis=1), probs

perm = rng.permutation(len(trace_df))
split = int(0.8 * len(trace_df))
train_idx = perm[:split]
test_idx = perm[split:]

train_df = trace_df.iloc[train_idx].reset_index(drop=True)
test_df = trace_df.iloc[test_idx].reset_index(drop=True)

X_train = train_df[shape_cols].to_numpy(dtype=float)
y_train = train_df["action_id"].to_numpy(dtype=int)
X_test = test_df[shape_cols].to_numpy(dtype=float)
y_test = test_df["action_id"].to_numpy(dtype=int)

X_train_s, X_test_s, mu, sd = standardize_train_test(X_train, X_test)
W_bridge, b_bridge, loss_history = train_multiclass_logreg(X_train_s, y_train, sample_weight=None, lr=0.08, epochs=800)
pred_test, _ = predict_logreg(X_test_s, W_bridge, b_bridge)

bridge_head_accuracy = float((pred_test == y_test).mean())
bridge_head_accuracy


In [ ]:

action_recall_rows = []
for action_id, action_name in enumerate(ACTION_NAMES):
    mask = y_test == action_id
    recall = float((pred_test[mask] == action_id).mean()) if mask.sum() else np.nan
    action_recall_rows.append({
        "action": action_name,
        "support": int(mask.sum()),
        "recall": recall,
    })

bridge_head_recall = pd.DataFrame(action_recall_rows)
bridge_head_recall


In [ ]:

plt.figure(figsize=(10, 4))
plt.plot(loss_history)
plt.title("Bridge Head Training Loss")
plt.xlabel("epoch")
plt.ylabel("cross entropy")
plt.show()



## Run a recurrent lattice simulation

Now we compare:

- **heuristic bridge CPU**
- **learned bridge CPU**

on the same initial lattice.


In [ ]:

def policy_action_from_shape(shape: dict, truth_slot: int, mode: str):
    if mode == "heuristic":
        action, gate_score, packet_pass = bridge_oracle(shape, truth_slot)
        return action, gate_score, packet_pass

    x = np.array([[shape[col] for col in shape_cols]], dtype=float)
    x_s = (x - mu) / sd
    pred, probs = predict_logreg(x_s, W_bridge, b_bridge)
    action = ID_TO_ACTION[int(pred[0])]
    gate_score, packet_pass = packet_gate(shape)
    return action, gate_score, packet_pass

def lattice_metrics(state: dict, action_grid: np.ndarray):
    H, W = state["truth"].shape
    pred = np.zeros((H, W), dtype=int)
    for i in range(H):
        for j in range(W):
            pred[i, j], _, _ = predict_slot_from_tail(state["tail"][i, j])

    accuracy = float((pred == state["truth"]).mean())
    closure_mean = float(state["closure"].mean())

    carry_err = []
    for i in range(H):
        for j in range(W):
            cls = int(state["truth"][i, j])
            carry_target = 0.5 * tail_prototypes[cls, 1] + 0.5 * tail_prototypes[cls, 2]
            carry_err.append(abs(state["carry"][i, j] - carry_target))

    action_flat = action_grid.flatten().tolist()
    promote_rate = float(np.mean([a == "promote" for a in action_flat]))
    hold_rate = float(np.mean([a == "hold" for a in action_flat]))
    retire_rate = float(np.mean([a == "retire" for a in action_flat]))
    grow_carry_rate = float(np.mean([a == "grow_carry" for a in action_flat]))

    return {
        "accuracy": accuracy,
        "closure_mean": closure_mean,
        "carry_error": float(np.mean(carry_err)),
        "promote_rate": promote_rate,
        "hold_rate": hold_rate,
        "retire_rate": retire_rate,
        "grow_carry_rate": grow_carry_rate,
        "trace_mean": float(state["trace"].mean()),
    }

def clone_state(seed_state: dict):
    return {
        k: np.copy(v) if isinstance(v, np.ndarray) else v
        for k, v in seed_state.items()
    }

def run_lattice(mode: str, steps: int = 20, height: int = 8, width: int = 8, seed_state: dict | None = None):
    if seed_state is None:
        state = init_lattice(height=height, width=width)
    else:
        state = clone_state(seed_state)

    rows = []
    H, W = state["truth"].shape

    for step in range(steps):
        action_grid = np.empty((H, W), dtype=object)

        for i in range(H):
            for j in range(W):
                shape = packet_shape_features(state, i, j)
                action, gate_score, packet_pass = policy_action_from_shape(shape, int(state["truth"][i, j]), mode)
                action_grid[i, j] = action

        rows.append({"step": step, "mode": mode, **lattice_metrics(state, action_grid)})
        state = lattice_step_with_actions(state, action_grid)

    return state, pd.DataFrame(rows)

seed_state = init_lattice(height=8, width=8)
heur_state, heur_metrics = run_lattice("heuristic", steps=20, seed_state=seed_state)
learn_state, learn_metrics = run_lattice("learned", steps=20, seed_state=seed_state)

sim_df = pd.concat([heur_metrics, learn_metrics], ignore_index=True)
sim_df.head()



## Simulation trajectories


In [ ]:

metrics = [
    "accuracy",
    "closure_mean",
    "carry_error",
    "promote_rate",
    "hold_rate",
    "retire_rate",
    "grow_carry_rate",
    "trace_mean",
]

for metric in metrics:
    plt.figure(figsize=(10, 4))
    for mode in ["heuristic", "learned"]:
        df = sim_df[sim_df["mode"] == mode]
        plt.plot(df["step"], df[metric], marker="o", label=mode)
    plt.title(metric.replace("_", " ").title())
    plt.xlabel("step")
    plt.ylabel(metric)
    plt.legend(loc="best")
    plt.show()



## Final comparison


In [ ]:

final_compare = pd.DataFrame([
    {
        "heuristic_final_accuracy": float(heur_metrics.iloc[-1]["accuracy"]),
        "learned_final_accuracy": float(learn_metrics.iloc[-1]["accuracy"]),
        "heuristic_final_closure": float(heur_metrics.iloc[-1]["closure_mean"]),
        "learned_final_closure": float(learn_metrics.iloc[-1]["closure_mean"]),
        "heuristic_final_carry_error": float(heur_metrics.iloc[-1]["carry_error"]),
        "learned_final_carry_error": float(learn_metrics.iloc[-1]["carry_error"]),
        "heuristic_final_promote_rate": float(heur_metrics.iloc[-1]["promote_rate"]),
        "learned_final_promote_rate": float(learn_metrics.iloc[-1]["promote_rate"]),
    }
])
final_compare



## Feature importance proxy


In [ ]:

importance = pd.DataFrame({
    "feature": shape_cols,
    "importance": np.abs(W_bridge).mean(axis=1),
}).sort_values("importance", ascending=False)

importance



## Practical readout

If this notebook works well, the software stack is now clear:

- **LLM**
  - proposal engine only

- **Inner CPU / packet cells**
  - recurrent packet state
  - local closure
  - local carry
  - local trace

- **Outer CPU / bridge policy**
  - govern promote / hold / retire / repair

- **Value layer**
  - late readout only

## What to do next

If v24 is useful:

1. Replace synthetic corruption with real runtime traces.
2. Keep the same action targets.
3. Keep the recurrent lattice state.
4. Keep the LLM outside the lattice as proposer only.
5. Only after that consider scaling the bridge head or adding adapters.

This is the nested-CPU lattice runtime.
